# Day 9: error analysis for spectra and measuring variability


From this notebook, we will figure out how to use the allspec database on SkyServer compute to: 

--review the format and contents of the allspec and spAll SDSS-V data files

--identify a high signal to noise quasar with multiple spectra

--measure data errors and a time-averaged spectrum

--inspect the time series to decide which if any parts of the spectrum are changing in intrinsic brightness above the level of the errors

Then for the lab, you will adapt this code to carry out a similar analysis for a different quasar of your choice.

In [ ]:
import os
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import astropy.io.fits
import astropy.coordinates
import fitsio
import sdss_access

matplotlib.rcParams['text.usetex'] = True
matplotlib.rcParams['font.size'] = 14

#### Do not edit the following code adapted from the SDSS allspec tutorial to load the database

In [ ]:
sdss_path = sdss_access.path.Path(release='dr19', verbose=True)
access = sdss_access.Access(release='dr19', verbose=True)

In [ ]:
# this step is slow give it time the allspec file is enormous
allspec_file = sdss_path.full('allspec', vers='1.0.1', release='dr19')
allspec_hdus = astropy.io.fits.open(allspec_file)
allspec = np.array(allspec_hdus[1].data)

## 1. Review: data contained in allspec

allspec is useful for finding spectra from *all types of objects*

good for matching locations on sky

does not have detailed information about redshifts, brightness, signal to noise

### 2. Review: data contained in spAll

spAll is for BOSS spectra for quasars specifically

has detailed information specific to this type of spectrum

very large file, the command below only loads the column names contained in spall_columns. There are many others and you can check the data model at https://data.sdss.org/datamodel/files/BOSS_SPECTRO_REDUX/RUN2D/spAll.html

### 3. Review: define a sample of quasar spectra using spAll

In [ ]:
# this step loads data from the spall file and so can take several minutes to run
spall_file = sdss_path.full('spAll', run2d='v6_1_3')

if not sdss_path.exists('',full=spall_file):
    # if the file does not exist locally, this code will download the data.
    access.remote()
    access.add('spAll', run2d='v6_1_3')
    access.set_stream()
    access.commit()
print(spall_file)

spall_columns = ['SDSS_ID', 'CARTON_TO_TARGET_PK', 'MJD', 'CLASS', 'SUBCLASS', 'Z', 'ZWARNING', 'SN_MEDIAN_ALL', 'PSFMAG']
spall = fitsio.read(spall_file, columns=spall_columns)

Here spall['Z'] contains redshifts, spall['PSFMAG'] contains ugriz magnitudes for the spectrum, and spall['SN_MEDIAN_ALL'] contains a summary signal to noise

#### here is a condition to find a sample of bright objects with high signal to noise spectra near a redshift of 0.5

In [ ]:
indx = np.where((spall['PSFMAG'][:,1] < 18.) & (spall['Z'] > 0.45) & (spall['Z'] < 0.55) & (spall['SN_MEDIAN_ALL'] > 30.))
sdssid_sample = spall['SDSS_ID'][indx]

#### we'll now find a list of all objects (by sdss_id) with between 5-25 spectra (this can take a few minutes)

In [ ]:
iallmatch_list = []

# for some list of matching objects, let's find all of those with many spectra
for sdssid in sdssid_sample:
    iallmatch = np.where(allspec['sdss_id'] == sdssid)[0]
    files = allspec['allspec_id'][iallmatch]
    
    # let's find one with somewhere around 10-20 spectra
    if (len(files) > 5) & (len(files) < 25):
        iallmatch_list.append(iallmatch)

### 4. Review: for some list from spAll, figure out how to load the spectra

To find files on disk, we need to use the online location ('sas_url') and then modify it to point to the local file system on SciServer.

The code with "download_files" is not needed when you are writing your own code and working on SciServer.

For the Lab, to pick your own object you could change the variable "obj" here to match some other object in the iallmatch_list variable.

In [ ]:
obj = 0
iallmatch = iallmatch_list[obj]

url_root = 'https://data.sdss.org/sas'
local_root = os.getenv('SAS_BASE_DIR')
spectrum_files = list()
download_files = list()

for p in allspec["sas_url"][iallmatch]:
    local_path = p.decode().replace(url_root, local_root)
    spectrum_files.append(local_path)
    if not os.path.exists(local_path):
        download_files.append(local_path)

if len(download_files) > 0:
    print("fetching files, please stand by")
    access.remote()
    for local_path in download_files:
        access.add_file(local_path, input_type='filepath')

    access.set_stream()

    # disable follow_symlinks
    access.commit(follow_symlinks=False)

Here then are the paths in the local SAS directory structure:

In [ ]:
for f in spectrum_files:
    print(f)

We can open one of the BOSS files up to see what it has in it. We'll first just look at what the HDUs are called.

Now we can plot and label our plot:

In [ ]:
# load all of the data files ('COADD' data extension)
boss_visits = []
mjds = []
# only using the last 20 since the data format changes
for file in spectrum_files[-15:-1]:
    spec_hdulist = astropy.io.fits.open(file)
    visit = np.array(spec_hdulist['COADD'].data)
    boss_visits.append(visit)
    mjds.append(spec_hdulist[0].header['MJD'])
    
# general object info
z = spec_hdulist[2].data['Z'][0]

In [ ]:
# Plot all the spectra
allmax = 0.0
for mjd, visit in zip(mjds, boss_visits):
    gd = visit['IVAR'] > 0
    gdmax = visit['MODEL'][gd].max()
    if(gdmax > allmax):
        allmax = gdmax

    plt.plot(10.**visit['LOGLAM'], visit['FLUX'], linewidth=1, label=str(mjd))

plt.ylim(np.array([-0.05, 1.3]) * allmax)
plt.xlabel(r'\rm Wavelength (Angstroms)')
plt.ylabel(r'$f_\lambda$ \rm ($10^{-17}$ erg cm$^{-2}$ s$^{-1}$ \AA$^{-1}$)')

### 5. Let's make a time-averaged spectrum to reduce the noise

In [ ]:
### define a variable to contain the co-added spectrum
spec_avg = np.zeros(len(boss_visits[0]['FLUX']))
for visit in boss_visits:
        spec_avg += visit['FLUX']
        
# now divide by number of files
spec_avg /= len(boss_visits)

# plot result
lam0=10**boss_visits[0]['LOGLAM']
plt.plot(lam0,spec_avg)
plt.ylim(np.array([-0.05, 1.3]) * allmax)

What are the broad emission lines that you see? You can use the redshift and the list here (from day01 notebook) to identify a few

### here are some useful atomic emission lines in quasar spectra (units of Angstrom in emitted frame):
O VI 1034

Ly $\alpha$ 1216

C IV 1549

C III 1908

Mg II 2799

H$\beta$ 4862 

H$\gamma$ 4361

H$\alpha$ 6564

### 6. How do we know if the object's brightness actually changed or if the spread is just noisy measurements?

let's estimate the errors in the data

In [ ]:
lam = 10**boss_visits[0]['LOGLAM']
flux0 = boss_visits[0]['FLUX']
# ivar is the inverse variance 1/sigma^2
errors0 = XX # add your code here to calculate sigma using the inverse variance boss_visits[0]['IVAR']

# first find a region of 50 channels on either side of 8500 Angstrom
XX

# now estimate the noise as the standard deviation (np.std) of the flux in your selected +/- 50 channel region
noise = XX

# now get the median value of the errors from the pipeline in the same part of the spectrum using errors0
pipeline_noise = XX
print('noise estimate and from pipeline: ',noise,pipeline_noise)

### 7. Make a continuum light curve (flux vs time)

Let's calculate the flux vs. time in some wavelength region along with error bars

In [ ]:
# let's make a continuum light curve at 8500 Angstrom with error bars from the scatter in the data
flux_8500 = []
flux_8500_err = []

for visit in boss_visits:
    lam = 10**visit['LOGLAM']
    flux = visit['FLUX']
    # write code to calculate the median flux +/- 50 indices around an observed wavelength of 8500 Angstrom
    # and the uncertainty in the value as the standard deviation (np.std) of those values
    XX
    
    flux_8500.append(XX)
    flux_8500_err.append(XX)

In [ ]:
# now plot the result vs the date (mjds) using plt.errorbar from matplotlib.pyplot
# you'll want to use linestyle='' and marker='o' to avoid drawing lines between data points
XX

### 8. Now check the brightness of a narrow emission line

In [ ]:
# plot the peak brightness of the OIII narrow emission line (spike near 7400 Angstrom observed) vs time
# do this by taking the median of 3 pixels on either side of the peak

In [ ]:
# this is the observed wavelength of OIII
lam_oiii=(1.+z)*5007.

# write code to identify a region 50 pixels on either side of the OIII line
XX

# now write a loop that calculates the line flux oiii_flux_approx and its errors oiii_errors
oiii_flux_approx = []
oiii_errors = []
for visit in boss_visits:
    # define the flux in the region +/- 50 indices on either side of the OIII spike
    flux_zoom = # flux over the region +/- 50 indices around OIII
    ivar_zoom = # IVAR over the same region
    
    # estimate the location of OIII as closest to flux_zoom and the value there
    oiii_peak = np.argmax(flux_zoom)
    oiii_peak_flux = flux_zoom[oiii_peak]
    # estimate the background around the line
    oiii_background=np.median(np.append(flux_zoom[:5],flux_zoom[-5:]))
    oiii_flux_t = oiii_peak_flux-oiii_background
    oiii_flux_approx.append(oiii_flux_t)
    oiii_errors.append(np.median(1./np.sqrt(ivar_zoom[oiii_peak-3:oiii_peak+3])))

### now plot the OIII flux vs. time with error bars

In [ ]:
# use matplotlib.pyplot.errorbar (plt.errorbar) to make a plot like above

### 9. Now check the brightness of the broad $H\alpha$ emission line

In [ ]:
# fit the H\alpha line with your lab 1 solution
# calculate the observed wavelength of H\alpha
lam_halpha = XX

# find the index location close to H\alpha and define a region +/- 250 pixels from the center
XX

# make a plot of the time-averaged spectrum in this region
plt.plot()

### Now use your Lab 1 / Lab 2 solutions to measure the line peak for each spectrum 

In [ ]:
# put your code here

In [ ]:
# plot the "A" parameter from the fit as a function of mjd with error bars

### What do you think about the 3 plots? Do any of the quantities appear to show evidence of intrinsic variability beyond that expected from the noise?